<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 그래프 학습 · 추천 시스템 · 07. GIN: 합 집계와 WL 표현력

## How Powerful are Graph Neural Networks?

- **원 논문:** [How Powerful are Graph Neural Networks?](https://openreview.net/forum?id=ryGs6iA5Km)
- **저자 / 발표:** Keyulu Xu, Weihua Hu, Jure Leskovec, Stefanie Jegelka · ICLR (2019)
- **난이도 / 예상 시간:** 중급 · 약 55분
- **선수 지식:** message passing, multiset, Weisfeiler–Lehman test
- **로컬 학습 데이터:** `data/field_curriculum/graph_recommendation.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** sum aggregator와 (1+epsilon) self term을 구현해 mean이 잃는 degree 정보를 보존함을 확인합니다.

**축소하거나 생략한 부분:** 9개 graph-classification benchmark 대신 한 로컬 그래프의 node multiset 표현력을 점검합니다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §4.1, Lemma 5 및 Corollary 6 | injective multiset sum 직관 | degree 구별 |
| §4.2, Eq. (4.1) | GIN update | epsilon self term |
| §4.3, Eq. (4.2) | layer별 graph readout | permutation invariance |
| Figure 2 | mean/max/sum 표현력 비교 | 고유 representation 수 |

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** A [N,N], X [N,D] → graph embedding [G,H] → logits [G,K]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

random.seed(7)
np.random.seed(7)
torch.manual_seed(7)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(7)

DATA_PATH = Path("data/field_curriculum/graph_recommendation.npz")
assert DATA_PATH.exists(), f"먼저 로컬 학습 데이터를 준비하세요: {DATA_PATH}"
raw = np.load(DATA_PATH, allow_pickle=False)
adjacency = torch.tensor(raw["adjacency"], dtype=torch.float32)
features = torch.tensor(raw["features"], dtype=torch.float32)
labels = torch.tensor(raw["labels"], dtype=torch.long)
train_mask = torch.tensor(raw["train_mask"], dtype=torch.bool)
test_mask = torch.tensor(raw["test_mask"], dtype=torch.bool)
assert adjacency.ndim == 2 and adjacency.shape[0] == adjacency.shape[1] == len(features)
assert torch.allclose(adjacency, adjacency.T) and not torch.diagonal(adjacency).any()
edge_count = int(adjacency.sum().item() / 2)
print(f"nodes={len(features)}, edges={edge_count}, features={features.shape[1]}")
print(ACCELERATOR.summary())
print(
    "graph sampling/topology preprocessing stays on CPU, "
    "while neural training uses DEVICE"
)

## 포트폴리오 해설: 핵심 수식에서 실행 코드까지

$$h_v^k=\operatorname{MLP}^k\!\left((1+\epsilon^k)h_v^{k-1}+\sum_{u\in
\mathcal N(v)}h_u^{k-1}\right)$$

- **기호 정의:** ε은 self 가중치, 합은 이웃 multiset, MLP는 injective map 근사입니다.
- **수식의 역할:** sum aggregation으로 mean이 지우는 이웃 중복도와 degree 정보를 보존합니다.
- **구현 이유:** 원문 전체를 축약하더라도 위 관계가 결과를 만드는 핵심이므로,
  이 수식을 전처리·`forward`·loss 경계로 나누어 직접 구현했습니다.
- **Task/코드 대응:** 두 GIN layer가 §4.2의 update를 수행하고,
  layer-wise sum readout과 graph prediction head가 §4.3을 구현합니다.
- **입출력 shape:** ego graph [N_g, F], [N_g, N_g] →
  layer states → concatenated graph readout → graph logit [1, C]
- **평가:** held-out ego-graph accuracy, readout 순열 불변성,
  sum/mean의 degree·multiset 분리
- **원문 대비 한계:** 한 로컬 graph의 ego-network를 graph sample로 사용하므로,
  원 논문의 9개 graph-classification benchmark와 직접 비교하지 않습니다.

코드를 읽을 때는 `Config → 전처리 → Module.forward → loss → train → evaluate`
순서로 추적하세요. 각 함수는 한 가지 책임만 갖도록 분리했습니다.

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §4.2, Eq. (4.1)
- **이 셀의 책임:** GIN update
- **Tensor shape 계약:** A [N,N], X [N,D] → graph embedding [G,H] → logits [G,K]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** epsilon self term. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def gin_aggregate(x: Tensor, adj: Tensor, epsilon: float = 0.0) -> Tensor:
    return (1 + epsilon) * x + adj @ x


ones = torch.ones(len(adjacency), 1)
gin_counts = gin_aggregate(ones, adjacency)
assert torch.allclose(gin_counts[:, 0], adjacency.sum(1) + 1)

### 구현 단계 3 · 논문 주장 확인 실험

- **원문 위치:** §4.1, Lemma 5 및 Corollary 6 / Figure 2
- **이 셀의 책임:** injective multiset sum 직관 / mean/max/sum 표현력 비교
- **Tensor shape 계약:** A [N,N], X [N,D] → graph embedding [G,H] → logits [G,K]
- **구현 이유:** 한 변수만 바꾸어 해당 메커니즘이 출력에 미치는 영향을 분리합니다.
- **완료 증거:** degree 구별 / 고유 representation 수. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
degree = adjacency.sum(1, keepdim=True)
mean_neighbors = (adjacency @ ones) / degree.clamp_min(1)
active = degree[:, 0] > 0
mean_unique = len(torch.unique(mean_neighbors[active]))
sum_unique = len(torch.unique((adjacency @ ones)[active]))
assert mean_unique == 1 and sum_unique > mean_unique
print({"mean_unique": mean_unique, "sum_unique": sum_unique})

### 구현 단계 4 · 평가·완료 증거

- **원문 위치:** §4.2, Eq. (4.1) / §4.3, Eq. (4.2)
- **이 셀의 책임:** GIN update / layer별 graph readout
- **Tensor shape 계약:** A [N,N], X [N,D] → graph embedding [G,H] → logits [G,K]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** epsilon self term / permutation invariance. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
mlps = nn.ModuleList(
    [
        nn.Sequential(
            nn.Linear(features.shape[1] if i == 0 else 10, 10),
            nn.ReLU(),
            nn.Linear(10, 10),
        )
        for i in range(3)
    ]
)
states = [features]
h = features
for mlp in mlps:
    h = mlp(gin_aggregate(h, adjacency, 0.1))
    states.append(h)
readouts = torch.stack([s.sum(0).norm() for s in states])
perm = torch.randperm(len(features))
p_adj = adjacency[perm][:, perm]
p_x = features[perm]
p_states = [p_x]
ph = p_x
for mlp in mlps:
    ph = mlp(gin_aggregate(ph, p_adj, 0.1))
    p_states.append(ph)
p_readouts = torch.stack([s.sum(0).norm() for s in p_states])
assert torch.allclose(readouts, p_readouts, atol=1e-5)
plt.plot(range(4), readouts.detach(), marker="o")
plt.xlabel("layer")
plt.ylabel("readout norm")
plt.title("GIN layer-wise readout")
plt.show()

### Task P1 · 전처리와 핵심 모델

먼저 graph 입력의 shape와 정규화 규칙을 고정하고, 논문의 핵심 연산을
`nn.Module`의 `forward`와 `compute_loss`로 분리합니다.

### 구현 단계 5 · 핵심 연산·모델

- **원문 위치:** §4.2, Eq. (4.1) / §4.3, Eq. (4.2)
- **이 셀의 책임:** GIN update / layer별 graph readout
- **Tensor shape 계약:** A [N,N], X [N,D] → graph embedding [G,H] → logits [G,K]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** epsilon self term / permutation invariance. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PortfolioConfig:
    hidden_dim: int = 12
    num_layers: int = 2
    learning_rate: float = 0.03
    steps: int = 18


def preprocess_graph(graph: Tensor) -> Tensor:
    return (graph > 0).to(graph.dtype)


def build_ego_graphs(
    inputs: Tensor,
    graph: Tensor,
    center_index: Tensor,
) -> list[tuple[Tensor, Tensor]]:
    examples = []
    for center in center_index.tolist():
        neighbors = torch.where(graph[center] > 0)[0]
        nodes = torch.cat(
            [torch.tensor([center]), neighbors],
        ).unique(sorted=True)
        ego_inputs = inputs[nodes].clone()
        ego_graph = graph[nodes][:, nodes].clone()
        examples.append((ego_inputs, ego_graph))
    return examples


class GinPortfolioModel(nn.Module):
    def __init__(
        self,
        input_dim: int,
        class_count: int,
        config: PortfolioConfig,
    ):
        super().__init__()
        self.config = config
        if config.num_layers < 2:
            raise ValueError("GIN requires at least two message-passing layers")
        self.epsilons = nn.ParameterList(
            [nn.Parameter(torch.tensor(0.0)) for _ in range(config.num_layers)]
        )
        self.mlps = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Linear(
                        input_dim if layer == 0 else config.hidden_dim,
                        config.hidden_dim,
                    ),
                    nn.ReLU(),
                    nn.Linear(config.hidden_dim, config.hidden_dim),
                    nn.ReLU(),
                )
                for layer in range(config.num_layers)
            ]
        )
        readout_dim = input_dim + config.num_layers * config.hidden_dim
        self.graph_classifier = nn.Linear(
            readout_dim,
            class_count,
        )

    def encode(self, inputs: Tensor, graph: Tensor) -> list[Tensor]:
        states = [inputs]
        hidden = inputs
        layer_parameters = zip(
            self.epsilons,
            self.mlps,
            strict=True,
        )
        for epsilon, mlp in layer_parameters:
            aggregated = gin_aggregate(hidden, graph, epsilon)
            hidden = mlp(aggregated)
            states.append(hidden)
        return states

    def graph_readout(self, inputs: Tensor, graph: Tensor) -> Tensor:
        layer_states = self.encode(inputs, graph)
        layer_readouts = [state.sum(dim=0) for state in layer_states]
        return torch.cat(layer_readouts, dim=0)

    def forward(self, inputs: Tensor, graph: Tensor) -> Tensor:
        readout = self.graph_readout(inputs, graph)
        return self.graph_classifier(readout).unsqueeze(0)

    def compute_loss(
        self,
        outputs: Tensor,
        targets: Tensor,
    ) -> Tensor:
        return F.cross_entropy(outputs, targets)

    def training_step(
        self,
        inputs: Tensor,
        graph: Tensor,
        targets: Tensor,
    ) -> Tensor:
        outputs = self(inputs, graph)
        return self.compute_loss(outputs, targets)

### Task P2 · objective와 update

`fit_portfolio_model`은 gradient가 흐르는 parameter와 한 step의 순서를
드러냅니다. 논문의 목적함수가 코드의 어느 tensor에 적용되는지 확인하세요.

### 구현 단계 6 · 목적함수·학습 update

- **원문 위치:** §4.2, Eq. (4.1)
- **이 셀의 책임:** GIN update
- **Tensor shape 계약:** A [N,N], X [N,D] → graph embedding [G,H] → logits [G,K]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** epsilon self term. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def fit_portfolio_model(
    model: GinPortfolioModel,
    graph_examples: list[tuple[Tensor, Tensor]],
    graph_targets: Tensor,
) -> list[float]:
    assert len(graph_examples) == len(graph_targets)
    optimizer = torch.optim.Adam(model.parameters(), lr=model.config.learning_rate)
    history = []
    model.train()
    for _ in range(model.config.steps):
        example_losses = []
        paired_examples = zip(
            graph_examples,
            graph_targets,
            strict=True,
        )
        for (inputs, graph), target in paired_examples:
            example_loss = model.training_step(
                inputs,
                graph,
                target.reshape(1),
            )
            example_losses.append(example_loss)
        loss = torch.stack(example_losses).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.append(float(loss.detach().cpu()))
    assert all(epsilon.grad is not None for epsilon in model.epsilons)
    assert model.graph_classifier.weight.grad is not None
    return history

### Task P3 · 평가와 한계 기록

loss와 별개의 논문 지표를 계산하고, 로컬 합성 graph에서 얻은 수치가
원 논문의 대규모 benchmark 결과를 대신하지 않는다는 점을 기록합니다.

### 구현 단계 7 · 평가·완료 증거

- **원문 위치:** §4.3, Eq. (4.2)
- **이 셀의 책임:** layer별 graph readout
- **Tensor shape 계약:** A [N,N], X [N,D] → graph embedding [G,H] → logits [G,K]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** permutation invariance. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def evaluate_portfolio_model(
    model: GinPortfolioModel,
    graph_examples: list[tuple[Tensor, Tensor]],
    graph_targets: Tensor,
) -> dict[str, float]:
    model.eval()
    with torch.no_grad():
        graph_logits = torch.cat(
            [model(inputs, graph) for inputs, graph in graph_examples],
            dim=0,
        )
        prediction = graph_logits.argmax(dim=1)
        accuracy = (prediction == graph_targets).float().mean()

        probe_inputs, probe_graph = graph_examples[0]
        permutation = torch.arange(
            len(probe_inputs) - 1,
            -1,
            -1,
            device=probe_inputs.device,
        )
        permuted_inputs = probe_inputs[permutation]
        permuted_graph = probe_graph[permutation][:, permutation]
        readout = model.graph_readout(probe_inputs, probe_graph)
        permuted_readout = model.graph_readout(
            permuted_inputs,
            permuted_graph,
        )
        permutation_error = (readout - permuted_readout).abs().max()
        assert torch.allclose(
            readout,
            permuted_readout,
            atol=1e-5,
        )
        assert torch.allclose(
            model(probe_inputs, probe_graph),
            model(permuted_inputs, permuted_graph),
            atol=1e-5,
        )

        layer_states = model.encode(probe_inputs, probe_graph)
        layer_norms = torch.stack([state.sum(dim=0).norm() for state in layer_states])

        degree_probe_graph = torch.tensor(
            [
                [0.0, 1.0, 0.0, 0.0],
                [1.0, 0.0, 1.0, 0.0],
                [0.0, 1.0, 0.0, 1.0],
                [0.0, 0.0, 1.0, 0.0],
            ],
            device=probe_inputs.device,
        )
        degree_probe_features = torch.ones(
            4,
            1,
            device=probe_inputs.device,
        )
        degree = degree_probe_graph.sum(dim=1, keepdim=True)
        mean_multiset = degree_probe_graph @ degree_probe_features
        mean_multiset = mean_multiset / degree.clamp_min(1.0)
        sum_multiset = gin_aggregate(
            degree_probe_features,
            degree_probe_graph,
            epsilon=0.0,
        )
        mean_unique = torch.unique(mean_multiset).numel()
        sum_unique = torch.unique(sum_multiset).numel()
        assert mean_unique == 1
        assert sum_unique > mean_unique
        assert torch.allclose(sum_multiset[:, 0], degree[:, 0] + 1.0)
    return {
        "heldout_ego_graph_accuracy": float(accuracy.cpu()),
        "readout_permutation_error": float(permutation_error.cpu()),
        "layer_readout_norm_mean": float(layer_norms.mean().cpu()),
        "layer_readout_count": float(len(layer_states)),
        "degree_multiset_unique_count": float(sum_unique),
    }


portfolio_train_index = torch.where(train_mask)[0]
portfolio_test_index = torch.where(test_mask)[0]
assert not torch.isin(portfolio_train_index, portfolio_test_index).any()
portfolio_config = PortfolioConfig()
portfolio_graph = preprocess_graph(adjacency)
portfolio_train_graphs = build_ego_graphs(
    features,
    portfolio_graph,
    portfolio_train_index,
)
portfolio_test_graphs = build_ego_graphs(
    features,
    portfolio_graph,
    portfolio_test_index,
)
portfolio_train_graphs = [
    ACCELERATOR.move(ego_inputs, ego_graph)
    for ego_inputs, ego_graph in portfolio_train_graphs
]
portfolio_test_graphs = [
    ACCELERATOR.move(ego_inputs, ego_graph)
    for ego_inputs, ego_graph in portfolio_test_graphs
]
portfolio_train_targets, portfolio_test_targets = ACCELERATOR.move(
    labels[portfolio_train_index],
    labels[portfolio_test_index],
)
portfolio_model = ACCELERATOR.move(
    GinPortfolioModel(
        features.shape[1],
        int(labels.max()) + 1,
        portfolio_config,
    )
)
portfolio_history = fit_portfolio_model(
    portfolio_model,
    portfolio_train_graphs,
    portfolio_train_targets,
)
portfolio_metrics = evaluate_portfolio_model(
    portfolio_model,
    portfolio_test_graphs,
    portfolio_test_targets,
)
assert np.isfinite(portfolio_history).all()
assert 0.0 <= portfolio_metrics["heldout_ego_graph_accuracy"] <= 1.0
assert portfolio_metrics["readout_permutation_error"] <= 1e-5
assert portfolio_metrics["layer_readout_count"] == portfolio_config.num_layers + 1
expected_readout_dim = features.shape[1]
expected_readout_dim += portfolio_config.num_layers * portfolio_config.hidden_dim
assert portfolio_model.graph_classifier.in_features == expected_readout_dim
assert portfolio_metrics["degree_multiset_unique_count"] > 1.0
print(portfolio_metrics)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 9개 graph-classification benchmark 대신 한 로컬 그래프의 node multiset 표현력을 점검합니다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.